# PocketLM GPU Smoke Test

Set the runtime to a free GPU, set `REPO_URL`, then run all cells. The final JSON includes the PLM-13 synthetic probe plus a real DeepSeek V3 FP8 remote-layer CUDA probe.

In [ ]:
REPO_URL = "https://github.com/iamlicht1f1-maker/pcketlm.git"
BRANCH = "plm-13-gpu-effective-speed"
WORKDIR = "/content/pcketlm"

assert REPO_URL.startswith("https://github.com/"), "Set REPO_URL to the real GitHub repo before running."

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path

token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token for private repo (leave blank for public repo): ")
clone_url = REPO_URL
if token and REPO_URL.startswith("https://"):
    clone_url = REPO_URL.replace("https://", f"https://{token}@", 1)

if Path(WORKDIR).exists():
    subprocess.check_call(["git", "-C", WORKDIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", WORKDIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", WORKDIR, "pull", "--ff-only", "origin", BRANCH])
else:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, WORKDIR])

os.chdir(WORKDIR)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytest"])
print("installed")

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device count", torch.cuda.device_count())
    print("device 0", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA is not available. In Colab, choose Runtime -> Change runtime type -> GPU.")

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "tools/deepseek_gpu_validate.py", "--require-cuda", "--json"])
subprocess.check_call([sys.executable, "-m", "pytest", "tests/gpu", "-q"])